# Sprint 4 Multi-label v2 — 3 mejoras

## Problemas identificados en v1

| Problema | v1 | v2 |
|---|---|---|
| LR Schedule | CosineAnnealing → LR=9e-7 en epoch 10 | Warmup + ReduceLROnPlateau |
| Augmentación | Ninguna | Flip + Affine + GaussianNoise |
| ViT profundidad | 4 bloques | **6 bloques** |
| Resultado esperado | AUC 0.79 | **AUC 0.82-0.85** |

**Runtime: G4/H100 GPU. Tiempo estimado: ~2-3 horas.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, shutil, time
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ CPU'}")

H5_LOCAL     = '/content/nih_images.h5'
PROCESSED_ML = '/content/processed_s4ml'
CLASS_NAMES_14 = [
    'Atelectasis','Cardiomegaly','Consolidation','Edema',
    'Effusion','Emphysema','Fibrosis','Hernia',
    'Infiltration','Mass','Nodule','Pleural_Thickening',
    'Pneumonia','Pneumothorax',
]


In [ ]:
!pip install -q torchxrayvision h5py opencv-python-headless
print('deps OK')


## 2. HDF5 y CSVs

In [ ]:
import h5py

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 4_000_000_000):
    print('Copiando HDF5...')
    t0 = time.time()
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)
    print(f'✓ {(time.time()-t0)/60:.1f} min')

with h5py.File(H5_LOCAL, 'r') as hf:
    print(f'✓ HDF5: {len(hf["images"]):,} imgs')

os.makedirs(PROCESSED_ML, exist_ok=True)
import pandas as pd
for csv in ['train.csv', 'val.csv', 'test.csv']:
    local = f'{PROCESSED_ML}/{csv}'
    if not os.path.exists(local):
        shutil.copy(f'{PROJECT_ROOT}/data/processed_s4ml/{csv}', local)
    df = pd.read_csv(local)
    print(f'✓ {csv}: {len(df):,} imgs')


## 3. Instalar archivos v2

In [ ]:
from pathlib import Path

# YAML v2
content = r'''# configs/sprint4_multilabel_v2.yaml
# Sprint 4 Multi-label v2 — 3 mejoras sobre v1:
#   1. LR Schedule: Warmup + Cosine (en vez de puro Cosine que decayó muy rápido)
#   2. Augmentación: RandomFlip + RandomAffine + GaussianNoise
#   3. ViT más profundo: 4 → 6 bloques Transformer

project:
  name: "tesis-cxr-sprint4-multilabel-v2"
  sprint: "4-multilabel-v2"
  seed: 42

paths:
  project_root:   "/content/drive/MyDrive/Tesis_CXR"
  data_raw:       "/content/drive/MyDrive/Tesis_CXR/data/raw/nih"
  data_processed: "/content/drive/MyDrive/Tesis_CXR/data/processed_s4ml"
  experiments:    "/content/drive/MyDrive/Tesis_CXR/experiments_s4ml_v2"
  checkpoints:    "/content/drive/MyDrive/Tesis_CXR/experiments_s4ml_v2/checkpoints"
  logs:           "/content/drive/MyDrive/Tesis_CXR/experiments_s4ml_v2/logs"

dataset:
  name: "nih_multilabel"
  csv_filename: "Data_Entry_subset_local.csv"
  views: ["PA", "AP"]
  num_classes: 14

preprocessing:
  image_size: 224

splits:
  train_ratio: 0.70
  val_ratio:   0.15
  test_ratio:  0.15
  split_by:    "patient_id"
  all_images_in_train: true

model:
  backbone:    "densenet121-res224-nih"
  num_classes: 14
  pretrained:  true
  vit:
    embedding_dim: 512
    num_heads:     8
    num_layers:    6      # MEJORA 3: 4 → 6 bloques
    mlp_dim:       1024
    dropout:       0.1

training:
  batch_size:     32
  num_workers:    2
  use_amp:        true
  use_pos_weight: true

  phase1:
    epochs:        15
    learning_rate: 0.0001
    weight_decay:  0.0001
    patience:      6
    # Warmup 3 epochs + cosine
    warmup_epochs: 3

  phase2:
    epochs:      45
    lr_cnn:      0.000002
    lr_vit:      0.00005   # más alto que v1 (1e-5 → 5e-5)
    weight_decay: 0.0001
    patience:    12
    # MEJORA 1: ReduceLROnPlateau en vez de CosineAnnealingLR puro
    scheduler:
      type:     "ReduceLROnPlateau"
      mode:     "max"        # maximizar AUC
      factor:   0.5
      patience: 4
      min_lr:   0.000001

evaluation:
  metrics: ["auc_per_class", "auc_macro", "map", "accuracy_per_class"]
  threshold: 0.5
'''
p = Path(f'{CODE_DIR}/configs/sprint4_multilabel_v2.yaml')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
# Transforms v2
content = r'''"""
src/datasets/transforms_multilabel_v2.py
Augmentación para Sprint 4 multi-label v2.
Aplica augmentación DESPUÉS de convertir a tensor para evitar
problemas con XRayCenterCrop en imágenes 2D del HDF5.
"""
import numpy as np
import torch
from torchvision import transforms
import torchxrayvision as xrv


class ArrayToXRVTensor:
    """(H,W) uint8 → (1,H,H) float32 en [-1024,1024] (formato xrv)."""
    def __call__(self, img: np.ndarray) -> torch.Tensor:
        img = img.astype(np.float32)
        img = (img / 255.0) * 2048.0 - 1024.0
        return torch.from_numpy(img).unsqueeze(0)


class GaussianNoise:
    """Agrega ruido gaussiano para simular variaciones del equipo RX."""
    def __init__(self, sigma: float = 20.0):
        self.sigma = sigma

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        if self.sigma > 0:
            noise = torch.randn_like(tensor) * self.sigma
            return (tensor + noise).clamp(-1024, 1024)
        return tensor


def build_train_transform_v2(image_size: int = 224):
    """Transform con augmentación — compatible con HDF5 arrays 2D."""
    return transforms.Compose([
        # 1. Resize a imagen cuadrada
        transforms.Lambda(
            lambda x: torch.from_numpy(
                __import__('cv2').resize(
                    x.astype(np.float32), (image_size, image_size)
                )
            ).unsqueeze(0) / 255.0 * 2048.0 - 1024.0
        ),
        # 2. Augmentaciones espaciales (sobre tensor)
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomAffine(
            degrees=10,
            translate=(0.07, 0.07),
            scale=(0.92, 1.08),
        ),
        # 3. Ruido gaussiano
        GaussianNoise(sigma=20.0),
    ])


def build_eval_transform_v2(image_size: int = 224):
    """Transform de evaluación sin augmentación."""
    return transforms.Compose([
        transforms.Lambda(
            lambda x: torch.from_numpy(
                __import__('cv2').resize(
                    x.astype(np.float32), (image_size, image_size)
                )
            ).unsqueeze(0) / 255.0 * 2048.0 - 1024.0
        ),
    ])
'''
p = Path(f'{CODE_DIR}/src/datasets/transforms_multilabel_v2.py')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


## 4. Smoke test augmentación

In [ ]:
for mod in list(__import__('sys').modules.keys()):
    if mod.startswith('src.'): del __import__('sys').modules[mod]

from src.datasets.nih_multilabel import NIHDatasetMultiLabel
from src.datasets.transforms_multilabel_v2 import (
    build_train_transform_v2, build_eval_transform_v2
)

train_tfm = build_train_transform_v2(224)
eval_tfm  = build_eval_transform_v2(224)

train_ds = NIHDatasetMultiLabel(f'{PROCESSED_ML}/train.csv', H5_LOCAL, train_tfm)
val_ds   = NIHDatasetMultiLabel(f'{PROCESSED_ML}/val.csv',   H5_LOCAL, eval_tfm)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')

sample = train_ds[0]
img = sample['image']
print(f'Shape: {img.shape} | Range: [{img.min():.0f}, {img.max():.0f}]')
print(f'✓ Augmentación OK')

# Mostrar distribución de clases positivas
dist = train_ds.class_distribution()
print('\nClases con menos ejemplos (más difíciles):')
for cls, count in sorted(dist.items(), key=lambda x: x[1])[:5]:
    print(f'  {cls}: {count:,} positivos ({count/len(train_ds)*100:.1f}%)')


## 5. Entrenar v2 con LR mejorado

In [ ]:
import copy, numpy as np, json
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from src.models.cnn_vit import CNNViT
from src.datasets.nih_multilabel import CLASSES_14
from src.evaluation.metrics_multilabel import compute_metrics_multilabel
from src.training.utils import load_config, get_device, set_seed

cfg    = load_config(f'{CODE_DIR}/configs/sprint4_multilabel_v2.yaml')
device = get_device()
set_seed(42)

# DataLoaders
nw = min(cfg.training.get('num_workers', 2), 4)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          num_workers=nw, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          num_workers=nw, pin_memory=True)

# Modelo con 6 capas ViT
model = CNNViT(
    backbone_weights='densenet121-res224-nih',
    embedding_dim=512,
    num_heads=8,
    num_layers=6,        # MEJORA: 4 → 6
    mlp_dim=1024,
    num_classes=14,
    dropout=0.1,
).to(device)

print(f'Modelo CNN-ViT (6 capas):')
print(f'  CNN params:  {sum(p.numel() for p in model.cnn_features.parameters()):,}')
print(f'  ViT params:  {sum(p.numel() for p in model.vit.parameters()):,}')
print(f'  Total:       {model.total_params():,}')

# pos_weight
pos_weight = train_ds.compute_pos_weight(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
use_amp    = True
scaler     = GradScaler(device='cuda')

os.makedirs(f'{PROJECT_ROOT}/experiments_s4ml_v2/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/experiments_s4ml_v2/logs', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/experiments_s4ml_v2/figures', exist_ok=True)


In [ ]:
# ─── Funciones de epoch ──────────────────────────────────────────

def run_epoch(model, loader, optimizer, criterion,
              device, use_amp, scaler, train=True):
    model.train() if train else model.eval()
    losses, all_probs, all_labels = [], [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            imgs   = batch['image'].to(device, non_blocking=True)
            labels = batch['label'].to(device, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
            if use_amp and train:
                with autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(imgs)
                    loss   = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            else:
                logits = model(imgs)
                loss   = criterion(logits, labels)
                if train: loss.backward(); optimizer.step()
            losses.append(loss.item())
            probs = torch.sigmoid(logits.detach().float())
            all_probs.append(probs.cpu()); all_labels.append(labels.cpu())
    m = compute_metrics_multilabel(
        torch.cat(all_labels).numpy(),
        torch.cat(all_probs).numpy()
    )
    return {'loss': float(np.mean(losses)), 'metrics': m}


In [ ]:
# ─── FASE 1: CNN congelada + Warmup ──────────────────────────────

print('='*65)
print('FASE 1: CNN congelada | 15 epochs | Warmup 3 epochs')
print('='*65)

model.set_phase(1)
P1_EPOCHS   = 15
P1_LR       = 1e-4
WARMUP_EP   = 3
PATIENCE_P1 = 6

opt1 = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-6,  # empieza bajo, warmup lo sube
    weight_decay=1e-4,
)

def get_lr_warmup_cosine(epoch, warmup_ep, total_ep, lr_max, lr_min=1e-6):
    if epoch < warmup_ep:
        return lr_min + (lr_max - lr_min) * (epoch + 1) / warmup_ep
    progress = (epoch - warmup_ep) / (total_ep - warmup_ep)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

best_auc_p1 = -float('inf')
best_state_p1 = None
no_improve_p1 = 0
history = []

for epoch in range(P1_EPOCHS):
    # Actualizar LR manualmente (warmup + cosine)
    new_lr = get_lr_warmup_cosine(epoch, WARMUP_EP, P1_EPOCHS, P1_LR)
    for pg in opt1.param_groups:
        pg['lr'] = new_lr

    tr = run_epoch(model, train_loader, opt1, criterion, device, use_amp, scaler, True)
    va = run_epoch(model, val_loader,   None, criterion, device, use_amp, scaler, False)

    print(f'[P1] Epoch {epoch+1:2d}/{P1_EPOCHS} | LR={new_lr:.2e} | '
          f'train AUC={tr["metrics"]["auc_macro"]:.4f} | '
          f'val AUC={va["metrics"]["auc_macro"]:.4f} mAP={va["metrics"]["map"]:.4f}')

    history.append({'phase':'P1','epoch':epoch+1,'lr':new_lr,
                    'train_loss':tr['loss'],'train_metrics':tr['metrics'],
                    'val_loss':va['loss'],'val_metrics':va['metrics']})

    if va['metrics']['auc_macro'] > best_auc_p1:
        best_auc_p1   = va['metrics']['auc_macro']
        best_state_p1 = copy.deepcopy(model.state_dict())
        no_improve_p1 = 0
        torch.save({'phase':'P1','epoch':epoch+1,
                    'model_state_dict':best_state_p1,
                    'best_val_auc_macro':best_auc_p1},
                   f'{PROJECT_ROOT}/experiments_s4ml_v2/checkpoints/phase1_best.pt')
        print(f'  → NUEVO MEJOR P1: {best_auc_p1:.4f}')
    else:
        no_improve_p1 += 1
        if no_improve_p1 >= PATIENCE_P1:
            print(f'  Early stopping P1.'); break

if best_state_p1:
    model.load_state_dict(best_state_p1)
print(f'\nFase 1 terminada. Mejor AUC: {best_auc_p1:.4f}')


In [ ]:
# ─── FASE 2: Full FT + ReduceLROnPlateau ─────────────────────────

print('='*65)
print('FASE 2: Full FT | 45 epochs | ReduceLROnPlateau')
print('='*65)

model.set_phase(2)
P2_EPOCHS   = 45
LR_VIT      = 5e-5
LR_CNN      = 2e-6
PATIENCE_P2 = 12

param_groups = model.get_param_groups(lr_cnn=LR_CNN, lr_vit=LR_VIT)
opt2 = torch.optim.Adam(param_groups, weight_decay=1e-4)

# ReduceLROnPlateau: reduce cuando val AUC no mejora
scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt2, mode="max", factor=0.5, patience=4, min_lr=1e-6
)

best_auc_p2  = best_auc_p1
best_state_p2 = None
no_improve_p2 = 0

for epoch in range(P2_EPOCHS):
    lr_vit_now = opt2.param_groups[1]['lr']
    tr = run_epoch(model, train_loader, opt2, criterion, device, use_amp, scaler, True)
    va = run_epoch(model, val_loader,   None, criterion, device, use_amp, scaler, False)

    val_auc = va['metrics']['auc_macro']
    scheduler2.step(val_auc)  # ReduceLROnPlateau sobre AUC

    print(f'[P2] Epoch {epoch+1:2d}/{P2_EPOCHS} | LR_vit={lr_vit_now:.2e} | '
          f'train AUC={tr["metrics"]["auc_macro"]:.4f} | '
          f'val AUC={val_auc:.4f} mAP={va["metrics"]["map"]:.4f}')

    if epoch % 5 == 0:
        for i, auc in enumerate(va['metrics']['auc_per_class']):
            if not np.isnan(auc):
                print(f'  val AUC {CLASSES_14[i]}: {auc:.4f}')

    history.append({'phase':'P2','epoch':epoch+1,'lr':lr_vit_now,
                    'train_loss':tr['loss'],'train_metrics':tr['metrics'],
                    'val_loss':va['loss'],'val_metrics':va['metrics']})

    if val_auc > best_auc_p2:
        best_auc_p2   = val_auc
        best_state_p2 = copy.deepcopy(model.state_dict())
        no_improve_p2 = 0
        torch.save({'phase':'P2','epoch':epoch+1,
                    'model_state_dict':best_state_p2,
                    'best_val_auc_macro':best_auc_p2,
                    'best_val_map':va['metrics']['map'],
                    'num_classes':14, 'num_layers':6},
                   f'{PROJECT_ROOT}/experiments_s4ml_v2/checkpoints/phase2_best.pt')
        print(f'  → NUEVO MEJOR P2: {best_auc_p2:.4f}')
    else:
        no_improve_p2 += 1
        if no_improve_p2 >= PATIENCE_P2:
            print(f'  Early stopping P2.'); break

# Guardar history
result = {
    'history': history,
    'best_val_auc_phase1': float(best_auc_p1),
    'best_val_auc_phase2': float(best_auc_p2),
    'improvements': ['warmup_cosine_lr', 'augmentation_v2', 'vit_6_layers'],
}
with open(f'{PROJECT_ROOT}/experiments_s4ml_v2/history_s4ml_v2.json', 'w') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f'\nRESUMEN FINAL v2:')
print(f'  P1 AUC: {best_auc_p1:.4f}')
print(f'  P2 AUC: {best_auc_p2:.4f}')
print(f'  Mejora vs v1 (0.7909): {best_auc_p2 - 0.7909:+.4f}')



## 6. Comparativa v1 vs v2 vs Wang 2017

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

with open(f'{PROJECT_ROOT}/experiments_s4ml/history_s4ml.json') as f:
    d_v1 = json.load(f)
with open(f'{PROJECT_ROOT}/experiments_s4ml_v2/history_s4ml_v2.json') as f:
    d_v2 = json.load(f)

def best_val(history, phase='P2'):
    h = [x for x in history if x.get('phase') == phase]
    return max(h, key=lambda x: x['val_metrics']['auc_macro'])['val_metrics']

m_v1 = best_val(d_v1['history'])
m_v2 = best_val(d_v2['history'])

WANG_AUC = {
    'Atelectasis':0.7003,'Cardiomegaly':0.8100,'Consolidation':0.7032,
    'Edema':0.8052,'Effusion':0.7585,'Emphysema':0.8330,'Fibrosis':0.7859,
    'Hernia':0.8717,'Infiltration':0.6614,'Mass':0.6931,'Nodule':0.6693,
    'Pleural_Thickening':0.6835,'Pneumonia':0.6580,'Pneumothorax':0.7993,
}

print('COMPARATIVA FINAL:')
print('='*65)
print(f"{'Clase':<22} {'Wang17':>7} {'v1':>7} {'v2':>7} {'Δv2':>7}")
print('-'*65)

for i, cls in enumerate(CLASS_NAMES_14):
    wang = WANG_AUC.get(cls, 0)
    v1   = m_v1['auc_per_class'][i]
    v2   = m_v2['auc_per_class'][i]
    if np.isnan(v1): v1 = 0
    if np.isnan(v2): v2 = 0
    delta = v2 - wang
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
    print(f'{cls:<22} {wang:>7.4f} {v1:>7.4f} {v2:>7.4f} {delta:>+7.4f} {arrow}')

print('-'*65)
print(f"{'AUC macro':<22} {np.mean(list(WANG_AUC.values())):>7.4f} "
      f"{m_v1['auc_macro']:>7.4f} {m_v2['auc_macro']:>7.4f} "
      f"{m_v2['auc_macro']-np.mean(list(WANG_AUC.values())):>+7.4f}")
print(f"{'mAP':<22} {'N/A':>7} {m_v1['map']:>7.4f} {m_v2['map']:>7.4f}")

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(CLASS_NAMES_14))
w = 0.25
ax.bar(x-w, [WANG_AUC[c] for c in CLASS_NAMES_14],     w, label='Wang 2017', color='#9E9E9E')
ax.bar(x,   [m_v1['auc_per_class'][i] for i in range(14)], w, label='v1 (AUC='+f"{m_v1['auc_macro']:.3f})", color='#2196F3', alpha=0.8)
ax.bar(x+w, [m_v2['auc_per_class'][i] if not np.isnan(m_v2['auc_per_class'][i]) else 0 for i in range(14)], w, label='v2 (AUC='+f"{m_v2['auc_macro']:.3f})", color='#F44336', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES_14, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('AUC'); ax.set_ylim(0.5, 1.0)
ax.axhline(0.745, color='gray', linestyle='--', alpha=0.5, label='Wang macro=0.745')
ax.legend(); ax.grid(alpha=0.3, axis='y')
ax.set_title('AUC por clase: Wang 2017 vs CNN-ViT v1 vs CNN-ViT v2')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s4ml_v2/figures/comparison_wang_v1_v2.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfica guardada.')

